# Lexsi DataAgent — Tiny Aya LoRA fine-tune (Track 3, Statement 6)

**What this notebook does:** fine-tunes CohereLabs Tiny Aya Global (3.35B) with LoRA (r=16, 4-bit NF4) on 26 verified question→SQL examples, evaluates base vs LoRA on 12 held-out task questions by EXECUTING the generated SQL, and uploads the adapter + datasets to Hugging Face.

**Honest scope:** 26 training examples, six narrow question patterns, one synthetic 10-order database. This is a prototype-scale adapter — it proves the AgentTune pipeline transfers to a trained model; it is NOT a general text-to-SQL model. All numbers below are measured, not asserted.

**Prerequisites (do these before Run All):**
1. Kaggle notebook with **GPU T4** accelerator ON (Settings → Accelerator → GPU T4).
2. Kaggle secret **`HF_TOKEN`** = a Hugging Face write token for `umarshareef0165` (Add-ons → Secrets). Needed for the gated Tiny Aya download + uploads. Without it, training cells 3–5 still run; upload cells print manual steps instead.
3. Tiny Aya licence: accept the licence at https://huggingface.co/CohereLabs/tiny-aya-global (same HF account). If the download 401s, the notebook falls back to `CohereLabs/tiny-aya-base` automatically.


In [ ]:
# 0 — Environment check
import torch, transformers, trl, peft, datasets, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__)
print("trl", trl.__version__, "| peft", peft.__version__, "| datasets", datasets.__version__)


In [ ]:
# 1 — Write the SFT data + eval set (embedded verbatim from the repo build)
import json, pathlib
train_rows = json.loads(r'''[
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-05 and 2026-10-06 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-05' AND '2026-10-06'",
  "question": "What is the revenue between 2026-10-05 and 2026-10-06 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-05' AND '2026-10-06'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-05 and 2026-10-10 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-05' AND '2026-10-10'",
  "question": "What is the revenue between 2026-10-05 and 2026-10-10 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-05' AND '2026-10-10'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-05 and 2026-10-13 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-05' AND '2026-10-13'",
  "question": "What is the revenue between 2026-10-05 and 2026-10-13 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-05' AND '2026-10-13'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-06 and 2026-10-08 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-06' AND '2026-10-08'",
  "question": "What is the revenue between 2026-10-06 and 2026-10-08 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-06' AND '2026-10-08'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-06 and 2026-10-11 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-06' AND '2026-10-11'",
  "question": "What is the revenue between 2026-10-06 and 2026-10-11 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-06' AND '2026-10-11'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-06 and 2026-10-14 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-06' AND '2026-10-14'",
  "question": "What is the revenue between 2026-10-06 and 2026-10-14 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-06' AND '2026-10-14'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-07 and 2026-10-10 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-07' AND '2026-10-10'",
  "question": "What is the revenue between 2026-10-07 and 2026-10-10 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-07' AND '2026-10-10'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-07 and 2026-10-13 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-07' AND '2026-10-13'",
  "question": "What is the revenue between 2026-10-07 and 2026-10-13 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-07' AND '2026-10-13'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-08 and 2026-10-10 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-08' AND '2026-10-10'",
  "question": "What is the revenue between 2026-10-08 and 2026-10-10 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-08' AND '2026-10-10'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-08 and 2026-10-13 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-08' AND '2026-10-13'",
  "question": "What is the revenue between 2026-10-08 and 2026-10-13 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-08' AND '2026-10-13'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-09 and 2026-10-11 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-09' AND '2026-10-11'",
  "question": "What is the revenue between 2026-10-09 and 2026-10-11 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-09' AND '2026-10-11'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-09 and 2026-10-14 for delivered orders?\nSQL:",
  "completion": " SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-09' AND '2026-10-14'",
  "question": "What is the revenue between 2026-10-09 and 2026-10-14 for delivered orders?",
  "sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-09' AND '2026-10-14'",
  "pattern": "revenue_in_range",
  "source": "expanded-date-range",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: List orders for Sara Malik.\nSQL:",
  "completion": " SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Sara Malik'",
  "question": "List orders for Sara Malik.",
  "sql": "SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Sara Malik'",
  "pattern": "orders_for_customer",
  "source": "expanded-customer",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: List orders for Omar Farooq.\nSQL:",
  "completion": " SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Omar Farooq'",
  "question": "List orders for Omar Farooq.",
  "sql": "SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Omar Farooq'",
  "pattern": "orders_for_customer",
  "source": "expanded-customer",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: List orders for Hina Raza.\nSQL:",
  "completion": " SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Hina Raza'",
  "question": "List orders for Hina Raza.",
  "sql": "SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Hina Raza'",
  "pattern": "orders_for_customer",
  "source": "expanded-customer",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Show delivered revenue grouped by product.\nSQL:",
  "completion": " SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "question": "Show delivered revenue grouped by product.",
  "sql": "SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "pattern": "revenue_by_product",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: How much revenue did each product bring in from delivered orders?\nSQL:",
  "completion": " SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "question": "How much revenue did each product bring in from delivered orders?",
  "sql": "SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "pattern": "revenue_by_product",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Break down delivered revenue by product.\nSQL:",
  "completion": " SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "question": "Break down delivered revenue by product.",
  "sql": "SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "pattern": "revenue_by_product",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Which customer spent the most on delivered orders?\nSQL:",
  "completion": " SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "question": "Which customer spent the most on delivered orders?",
  "sql": "SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "pattern": "top_customer",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Who is our biggest customer by delivered revenue?\nSQL:",
  "completion": " SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "question": "Who is our biggest customer by delivered revenue?",
  "sql": "SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "pattern": "top_customer",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Name the customer with the highest delivered spend.\nSQL:",
  "completion": " SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "question": "Name the customer with the highest delivered spend.",
  "sql": "SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "pattern": "top_customer",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the mean value of a delivered order?\nSQL:",
  "completion": " SELECT AVG(o.quantity * p.unit_price) AS avg_value FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered'",
  "question": "What is the mean value of a delivered order?",
  "sql": "SELECT AVG(o.quantity * p.unit_price) AS avg_value FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered'",
  "pattern": "avg_order_value",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: On average, how much is a delivered order worth?\nSQL:",
  "completion": " SELECT AVG(o.quantity * p.unit_price) AS avg_value FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered'",
  "question": "On average, how much is a delivered order worth?",
  "sql": "SELECT AVG(o.quantity * p.unit_price) AS avg_value FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered'",
  "pattern": "avg_order_value",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Compute the average delivered order value.\nSQL:",
  "completion": " SELECT AVG(o.quantity * p.unit_price) AS avg_value FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered'",
  "question": "Compute the average delivered order value.",
  "sql": "SELECT AVG(o.quantity * p.unit_price) AS avg_value FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered'",
  "pattern": "avg_order_value",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: How many pending orders are there?\nSQL:",
  "completion": " SELECT COUNT(*) AS n FROM orders WHERE status = 'pending'",
  "question": "How many pending orders are there?",
  "sql": "SELECT COUNT(*) AS n FROM orders WHERE status = 'pending'",
  "pattern": "count_by_status",
  "source": "paraphrase",
  "verified_by_execution": true
 },
 {
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Count the delivered orders.\nSQL:",
  "completion": " SELECT COUNT(*) AS n FROM orders WHERE status = 'delivered'",
  "question": "Count the delivered orders.",
  "sql": "SELECT COUNT(*) AS n FROM orders WHERE status = 'delivered'",
  "pattern": "count_by_status",
  "source": "paraphrase",
  "verified_by_execution": true
 }
]''')
eval_rows = json.loads(r'''[
 {
  "id": "T01",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the total revenue by product for delivered orders?\nSQL:",
  "question": "What is the total revenue by product for delivered orders?",
  "reference_sql": "SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "expected": "Office Chair: 960.0, Desk Lamp: 350.0, Laptop Stand: 135.0, Notebook Pack: 240.0",
  "must_contain": [
   "Office Chair",
   "960"
  ]
 },
 {
  "id": "T02",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Who is the top customer by delivered spend?\nSQL:",
  "question": "Who is the top customer by delivered spend?",
  "reference_sql": "SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "expected": "Bilal Ahmed (980.0: 600 + 140 + 240)",
  "must_contain": [
   "Bilal Ahmed",
   "980"
  ]
 },
 {
  "id": "T03",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: How many orders have status pending?\nSQL:",
  "question": "How many orders have status pending?",
  "reference_sql": "SELECT COUNT(*) AS n FROM orders WHERE status = 'pending'",
  "expected": "2",
  "must_contain": [
   "Count: 2"
  ]
 },
 {
  "id": "T04",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: How many orders have status delivered?\nSQL:",
  "question": "How many orders have status delivered?",
  "reference_sql": "SELECT COUNT(*) AS n FROM orders WHERE status = 'delivered'",
  "expected": "7",
  "must_contain": [
   "Count: 7"
  ]
 },
 {
  "id": "T05",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: How many orders have status cancelled?\nSQL:",
  "question": "How many orders have status cancelled?",
  "reference_sql": "SELECT COUNT(*) AS n FROM orders WHERE status = 'cancelled'",
  "expected": "1",
  "must_contain": [
   "Count: 1"
  ]
 },
 {
  "id": "T06",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the average order value for delivered orders?\nSQL:",
  "question": "What is the average order value for delivered orders?",
  "reference_sql": "SELECT AVG(o.quantity * p.unit_price) AS avg_value FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered'",
  "expected": "240.71",
  "must_contain": [
   "240.71"
  ]
 },
 {
  "id": "T07",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-05 and 2026-10-09 for delivered orders?\nSQL:",
  "question": "What is the revenue between 2026-10-05 and 2026-10-09 for delivered orders?",
  "reference_sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-05' AND '2026-10-09'",
  "expected": "1115.0 (240 + 600 + 135 + 140)",
  "must_contain": [
   "1115"
  ]
 },
 {
  "id": "T08",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: List orders for Bilal Ahmed.\nSQL:",
  "question": "List orders for Bilal Ahmed.",
  "reference_sql": "SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Bilal Ahmed'",
  "expected": "3 orders (ids 2, 5, 9)",
  "must_contain": [
   "Notebook Pack"
  ]
 },
 {
  "id": "T09",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: List orders for Ayesha Khan.\nSQL:",
  "question": "List orders for Ayesha Khan.",
  "reference_sql": "SELECT o.id, p.name AS product, o.quantity, o.order_date, o.status FROM orders o JOIN products p ON p.id = o.product_id JOIN customers c ON c.id = o.customer_id WHERE c.name = 'Ayesha Khan'",
  "expected": "3 orders (ids 1, 3, 10)",
  "must_contain": [
   "Office Chair"
  ]
 },
 {
  "id": "T10",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the total revenue by product for delivered orders?\nSQL:",
  "question": "What is the total revenue by product for delivered orders?",
  "reference_sql": "SELECT p.name AS product, SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY p.name ORDER BY revenue DESC",
  "expected": "same as T01 (repeat for stability)",
  "must_contain": [
   "Office Chair",
   "960"
  ]
 },
 {
  "id": "T11",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: What is the revenue between 2026-10-10 and 2026-10-13 for delivered orders?\nSQL:",
  "question": "What is the revenue between 2026-10-10 and 2026-10-13 for delivered orders?",
  "reference_sql": "SELECT SUM(o.quantity * p.unit_price) AS revenue FROM orders o JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' AND o.order_date BETWEEN '2026-10-10' AND '2026-10-13'",
  "expected": "570.0 (120 + 210 + 240; cancelled order excluded)",
  "must_contain": [
   "570"
  ]
 },
 {
  "id": "T12",
  "prompt": "Write a read-only SQL query for the question below using this schema.\n\nSchema:\ncustomers(id, name, city, segment)\nproducts(id, name, category, unit_price)\norders(id, customer_id, product_id, quantity, order_date, status)\n\nQuestion: Who is the top customer by delivered spend?\nSQL:",
  "question": "Who is the top customer by delivered spend?",
  "reference_sql": "SELECT c.name AS customer, SUM(o.quantity * p.unit_price) AS total FROM orders o JOIN customers c ON c.id = o.customer_id JOIN products p ON p.id = o.product_id WHERE o.status = 'delivered' GROUP BY c.name ORDER BY total DESC LIMIT 1",
  "expected": "same as T02 (repeat for stability)",
  "must_contain": [
   "Bilal Ahmed",
   "980"
  ]
 }
]''')
seed = json.loads(r'''{
 "customers": [
  [
   1,
   "Ayesha Khan",
   "Faisalabad",
   "retail"
  ],
  [
   2,
   "Bilal Ahmed",
   "Lahore",
   "wholesale"
  ],
  [
   3,
   "Sara Malik",
   "Karachi",
   "retail"
  ],
  [
   4,
   "Omar Farooq",
   "Islamabad",
   "wholesale"
  ],
  [
   5,
   "Hina Raza",
   "Faisalabad",
   "retail"
  ]
 ],
 "products": [
  [
   1,
   "Office Chair",
   "furniture",
   120.0
  ],
  [
   2,
   "Laptop Stand",
   "electronics",
   45.0
  ],
  [
   3,
   "Notebook Pack",
   "stationery",
   12.0
  ],
  [
   4,
   "Desk Lamp",
   "electronics",
   35.0
  ]
 ],
 "orders": [
  [
   1,
   1,
   1,
   2,
   "2026-10-05",
   "delivered"
  ],
  [
   2,
   2,
   1,
   5,
   "2026-10-06",
   "delivered"
  ],
  [
   3,
   1,
   2,
   3,
   "2026-10-07",
   "delivered"
  ],
  [
   4,
   3,
   3,
   10,
   "2026-10-08",
   "pending"
  ],
  [
   5,
   2,
   4,
   4,
   "2026-10-09",
   "delivered"
  ],
  [
   6,
   4,
   2,
   2,
   "2026-10-10",
   "cancelled"
  ],
  [
   7,
   5,
   1,
   1,
   "2026-10-11",
   "delivered"
  ],
  [
   8,
   3,
   4,
   6,
   "2026-10-12",
   "delivered"
  ],
  [
   9,
   2,
   3,
   20,
   "2026-10-13",
   "delivered"
  ],
  [
   10,
   1,
   4,
   1,
   "2026-10-14",
   "pending"
  ]
 ],
 "schema": "\nCREATE TABLE customers (\n    id INTEGER PRIMARY KEY,\n    name TEXT NOT NULL,\n    city TEXT NOT NULL,\n    segment TEXT NOT NULL\n);\nCREATE TABLE products (\n    id INTEGER PRIMARY KEY,\n    name TEXT NOT NULL,\n    category TEXT NOT NULL,\n    unit_price REAL NOT NULL\n);\nCREATE TABLE orders (\n    id INTEGER PRIMARY KEY,\n    customer_id INTEGER NOT NULL REFERENCES customers(id),\n    product_id INTEGER NOT NULL REFERENCES products(id),\n    quantity INTEGER NOT NULL,\n    order_date TEXT NOT NULL,\n    status TEXT NOT NULL\n);\n"
}''')
pathlib.Path("lexsi-dataagent-sft-train.jsonl").write_text(
    "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in train_rows))
pathlib.Path("lexsi-dataagent-sft-eval.json").write_text(
    json.dumps(eval_rows, indent=2, ensure_ascii=False))
pathlib.Path("seed.json").write_text(json.dumps(seed, indent=1))
print(f"train: {len(train_rows)} examples | eval: {len(eval_rows)} tasks")
print("leak check:", len({r['question'] for r in train_rows} & {r['question'] for r in eval_rows}), "eval questions in train (must be 0)")


In [ ]:
# 2 — Load base model (4-bit NF4) + tokenizer
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret: present")
except Exception as e:
    print("HF_TOKEN secret: MISSING —", e)

BASE_CANDIDATES = ["CohereLabs/tiny-aya-global", "CohereLabs/tiny-aya-base"]
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model = tok = BASE_ID = None
for cand in BASE_CANDIDATES:
    try:
        tok = AutoTokenizer.from_pretrained(cand, token=HF_TOKEN)
        model = AutoModelForCausalLM.from_pretrained(
            cand, quantization_config=bnb, device_map="auto", token=HF_TOKEN)
        BASE_ID = cand
        print("loaded base:", cand)
        break
    except Exception as e:
        print("FAILED", cand, "->", str(e)[:200])
assert model is not None, "could not load any Tiny Aya base model"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model.config.use_cache = False


In [ ]:
# 3 — Eval helpers + BASE-model eval (run BEFORE training)
import re, sqlite3, json, pathlib
from peft import PeftModel

SCHEMA = seed["schema"]
def open_db():
    conn = sqlite3.connect(":memory:"); conn.row_factory = sqlite3.Row
    conn.executescript(SCHEMA)
    conn.executemany("INSERT INTO customers VALUES (?,?,?,?)", [tuple(r) for r in seed["customers"]])
    conn.executemany("INSERT INTO products VALUES (?,?,?,?)", [tuple(r) for r in seed["products"]])
    conn.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", [tuple(r) for r in seed["orders"]])
    conn.commit(); return conn

def extract_sql(text):
    m = re.search(r"SELECT\b.*?;", text, re.I | re.S)
    if m: return m.group(0).strip()
    m = re.search(r"SELECT\b.*", text, re.I | re.S)
    return m.group(0).strip() if m else ""

def score_sql(sql, must):
    # mirrors the prototype harness: execute, format like the agent, check must_contain
    if not sql: return False, "no SQL extracted"
    conn = open_db()
    try:
        rows = [dict(r) for r in conn.execute(sql).fetchall()]
    except Exception as e:
        return False, f"SQL error: {e}"
    if not rows: return False, "empty result"
    first = rows[0]
    if "revenue" in first:
        ans = ", ".join(f"{r.get('product','')}: {r['revenue']}" for r in rows)
    elif "total" in first:
        ans = f"{first['customer']} ({first['total']})"
    elif "n" in first:
        ans = f"Count: {first['n']}."
    elif "avg_value" in first:
        ans = f"{first['avg_value']:.2f}"
    else:
        ans = str(rows)
    ok = all(s in ans for s in must)
    return ok, ans

def run_eval(m, label):
    results = []
    for task in eval_rows:
        ids = tok(task["prompt"], return_tensors="pt").to(m.device)
        out = m.generate(**ids, max_new_tokens=120, do_sample=False,
                         pad_token_id=tok.eos_token_id)
        gen = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        sql = extract_sql(gen)
        ok, detail = score_sql(sql, task["must_contain"])
        results.append({"id": task["id"], "question": task["question"],
                        "generated_sql": sql, "correct": bool(ok),
                        "detail": detail, "raw_generation": gen[:500]})
        print(f"[{label}] {task['id']} correct={ok} :: {sql[:110]}")
    acc = sum(r["correct"] for r in results) / len(results)
    print(f"[{label}] accuracy: {acc:.3f} ({sum(r['correct'] for r in results)}/{len(results)})")
    return results, acc

base_results, base_acc = run_eval(model, "base")


In [ ]:
# 4 — LoRA SFT (TRL SFTTrainer, prompt-completion, prompt masked)
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ds = Dataset.from_list([{"prompt": r["prompt"], "completion": r["completion"]}
                        for r in train_rows])
peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                      "gate_proj", "up_proj", "down_proj"])
cfg = SFTConfig(output_dir="./lora-out", per_device_train_batch_size=4,
                gradient_accumulation_steps=2, num_train_epochs=3,
                learning_rate=2e-4, logging_steps=5, save_strategy="no",
                fp16=True, max_length=512, packing=False, report_to=[])
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds,
                     peft_config=peft_cfg)
import time; t0 = time.time()
trainer.train()
print(f"training took {time.time()-t0:.0f}s")
print("final loss:", trainer.state.log_history[-1])
trainer.save_model("./adapter")
tok.save_pretrained("./adapter")
pathlib.Path("./adapter/BASE_MODEL.txt").write_text(BASE_ID + "\n")
print("adapter saved to ./adapter")


In [ ]:
# 5 — LoRA eval: reload the SAVED adapter on a fresh base, re-run 12 tasks
import gc
del model, trainer
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM
fresh = AutoModelForCausalLM.from_pretrained(
    BASE_ID, quantization_config=bnb, device_map="auto", token=HF_TOKEN)
fresh.config.use_cache = False
lora = PeftModel.from_pretrained(fresh, "./adapter")
lora_results, lora_acc = run_eval(lora, "lora")
report = {"base_model": BASE_ID, "base_accuracy": base_acc,
          "lora_accuracy": lora_acc, "n_tasks": len(eval_rows),
          "base": base_results, "lora": lora_results}
pathlib.Path("lexsi-dataagent-model-eval.json").write_text(json.dumps(report, indent=2))
print(f"\nHEADLINE: base {base_acc:.3f} -> LoRA {lora_acc:.3f} on {len(eval_rows)} held-out tasks")
print("saved lexsi-dataagent-model-eval.json")


In [ ]:
# 6 — Upload adapter + datasets to Hugging Face + create collections
# Needs HF_TOKEN (Kaggle secret). If missing, this cell prints manual steps.
from huggingface_hub import HfApi
import pathlib, shutil, json

USER = "umarshareef0165"
ADAPTER_REPO = f"{USER}/lexsi-dataagent-tiny-aya-lora"
SFT_REPO = f"{USER}/lexsi-dataagent-sft"
TRAJ_REPO = f"{USER}/lexsi-dataagent-trajectories"
if not HF_TOKEN:
    print("NO HF_TOKEN — manual upload steps:")
    print("1. huggingface_hub login with a write token for", USER)
    print("2. adapter ->", ADAPTER_REPO, " (folder ./adapter + model card)")
    print("3. sft files ->", SFT_REPO, " (train jsonl + eval json + this eval json)")
    print("4. outputs/*.json/jsonl ->", TRAJ_REPO)
    print("5. create 2 collections (models + datasets) and add the repos")
else:
    api = HfApi(token=HF_TOKEN)
    # model card into adapter folder
    pathlib.Path("./adapter/README.md").write_text('''---
license: cc-by-nc-4.0
base_model: CohereLabs/tiny-aya-global
tags:
- lexsi-hackathon
- track-3
- lora
---

# Lexsi DataAgent — Tiny Aya LoRA (Track 3, Statement 6)

LoRA adapter (r=16) fine-tuned on 26 verified question-to-SQL examples for a
read-only business-data agent (synthetic SQLite shop database: 5 customers,
4 products, 10 orders). The adapter teaches Tiny Aya Global to emit a single
read-only SELECT for six narrow question patterns (revenue by product, top
customer, order counts by status, average order value, revenue in a date
range, orders for a customer).

Training data: `umarshareef0165/lexsi-dataagent-sft` (every example's SQL was
executed against the seed database and verified before training). Evaluation:
12 held-out task questions (exact eval questions never appear in training),
base vs LoRA, scored by executing the generated SQL and checking the answer
— see `lexsi-dataagent-model-eval.json` in the dataset repo and the
write-up. This is a prototype-scale adapter (26 examples); it is not a
general text-to-SQL model, and that limit is stated in the write-up.
''')
    api.create_repo(ADAPTER_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path="./adapter", repo_id=ADAPTER_REPO, repo_type="model")
    print("adapter:", f"https://huggingface.co/{ADAPTER_REPO}")
    # datasets
    for repo, files in [
        (SFT_REPO, ["lexsi-dataagent-sft-train.jsonl", "lexsi-dataagent-sft-eval.json",
                    "lexsi-dataagent-model-eval.json"]),
        (TRAJ_REPO, None)]:
        api.create_repo(repo, repo_type="dataset", exist_ok=True)
        if files:
            for f in files:
                api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=repo, repo_type="dataset")
        print("dataset:", f"https://huggingface.co/datasets/{repo}")
    # NOTE: trajectory/eval-log files live in the repo outputs/ — upload them
    # from the GitHub repo (raw) in a follow-up cell, or add them here if present.
    # collections
    for title, desc, items in [
        ("Lexsi DataAgent — models", "Tiny Aya LoRA adapter for the Lexsi DataAgent (Track 3).",
         [(ADAPTER_REPO, "model")]),
        ("Lexsi DataAgent — datasets", "Trajectories, eval logs and SFT data for the Lexsi DataAgent (Track 3).",
         [(SFT_REPO, "dataset"), (TRAJ_REPO, "dataset")])]:
        col = api.create_collection(title=title, namespace=USER, description=desc, exist_ok=True)
        slug = col.slug
        for item_id, item_type in items:
            api.add_collection_item(collection_slug=slug, item_id=item_id, item_type=item_type)
        print("collection:", f"https://huggingface.co/collections/{USER}/{slug}")
print("DONE")


## Next steps (after this notebook)
1. Copy the headline numbers (base → LoRA accuracy) into the write-up article's Results table.
2. Upload `outputs/trajectories.jsonl`, `outputs/eval_results.json`, `outputs/auditkit_eval.json` to the trajectories dataset repo (from the GitHub repo or locally) + add to the datasets collection.
3. Paste the collection URLs + abstract into the Lexsi submit form (https://hackathon.lexsi.ai/submit — verify in a browser; the text fetch 404s) and submit ONLY with Umar's approval.
